# Module 4 – Bright spots, hot spots, yield gaps and crop WP gaps

**Last update: 19-Apr-2026**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wateraccounting/WaPORIPA/blob/main/Notebooks_v1.0/Module_4_BrightHotSpots_Yield_CWP_IPA_Gaps.ipynb)

This notebook covers two complementary analyses:

**A – Yield & CWP gaps** (absolute gaps to target, saved as maps)

**B – Bright / Hot spot identification** using two approaches:
1. *Crop-only*: crop yield + crop water productivity (cWP)
2. *Multi-indicator (weighted)*: yield + cWP + adequacy + beneficial fraction

For each approach the notebook:
- Derives targets from the upper percentile of each raster (no fixed gap thresholds)
- Produces **panel maps** showing – for each indicator – pixels that meet the bright/hot criterion, plus a final panel showing the **intersection** of all indicators
- Saves all outputs as GeoTIFF and provides a download zip


**=============================================================================================**

![title](https://github.com/wateraccounting/WAPORWP/blob/master/Notebooks/img/Fig5.PNG?raw=true)

**=============================================================================================**

## 1. Install and import packages

In [ ]:
!pip install -q rioxarray


In [ ]:
import os, glob, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib import colors
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import rioxarray
from IPython.display import display


## 2. Upload and unzip data

In [ ]:
from google.colab import files
uploaded = files.upload()   # Upload Module_3a_Yield_WP.zip and Module_3b_IPA.zip


In [ ]:
zip_files = [
    "/content/Module_3a_Yield_WP.zip",
    "/content/Module_3b_IPA.zip",
]

for zip_path in zip_files:
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall("/content")
        print("Unzipped:", zip_path)
    else:
        print("NOT FOUND:", zip_path)


## 3. User settings

> **Edit this cell to customise the analysis.**
>
> * `bright_percentile` / `hot_percentile` – define relative thresholds.  
> * `target_mode` – `"percentile"` (recommended) or `"manual"`.  
> * `weights` – adjust per-indicator importance for the multi-indicator analysis.  
>   Set any weight to **0** to exclude that indicator.  Default = equal weight.


In [ ]:
# ============================================================
# USER SETTINGS  –  edit here
# ============================================================

# --- Folder layout (matches Module 3a / 3b outputs) ---------
dir_data = "/content/content/output"    # root of unzipped data
dir_out  = "/content/output"            # where results are written

# --- Percentile thresholds ----------------------------------
bright_percentile = 90    # pixels >= this percentile = bright spot candidate
hot_percentile    = 10    # pixels <= this percentile = hot spot candidate

# --- Gap-target mode ----------------------------------------
# "percentile" -> target = bright_percentile of each raster (recommended)
# "manual"     -> use manual_target_* values below
target_mode = "percentile"

# Manual targets (only used when target_mode = "manual")
manual_target_yield = None   # e.g. 8.0  ton/ha
manual_target_cwp   = None   # e.g. 1.5  kg/m³

# --- Weights for multi-indicator analysis -------------------
# Default = equal weight (1.0 each).
# Increase a value to make that indicator more influential.
# Set to 0 to exclude an indicator from the weighted analysis.
weights = {
    "yield"              : 1.0,
    "cwp"                : 1.0,
    "adequacy"           : 1.0,
    "beneficial_fraction": 1.0,
}

# --- Optional strict mask -----------------------------------
strict_positive_mask = False   # True -> values <= 0 are set to NaN

# ============================================================
# Derived paths  –  no need to edit below this line
# ============================================================
yield_folder      = os.path.join(dir_data, "yield_season")
cwp_folder        = os.path.join(dir_data, "Cwp_season")
adequacy_folder   = os.path.join(dir_data, "Adequacy")
beneficial_folder = os.path.join(dir_data, "Beneficial_fraction")

out_targets       = os.path.join(dir_out, "Targets")
out_gaps          = os.path.join(dir_out, "Gaps")
out_crop_spots    = os.path.join(dir_out, "BrightHotSpots_Crop")
out_weighted_spots= os.path.join(dir_out, "BrightHotSpots_Weighted")

for d in [out_targets, out_gaps, out_crop_spots, out_weighted_spots]:
    os.makedirs(d, exist_ok=True)

print("Input folders:")
for label, folder in [("yield", yield_folder), ("cWP", cwp_folder),
                       ("adequacy", adequacy_folder), ("beneficial_fraction", beneficial_folder)]:
    exists = os.path.isdir(folder)
    print(f"  {label:<22}: {folder}  [{'OK' if exists else 'MISSING'}]")

print(f"\nBright percentile : {bright_percentile}")
print(f"Hot percentile    : {hot_percentile}")
print(f"Target mode       : {target_mode}")
print(f"Weights           : {weights}")


## 4. Helper functions

In [ ]:
# --- File utilities -----------------------------------------------------------

def sorted_tifs(folder):
    return sorted(glob.glob(os.path.join(folder, "*.tif")))

def extract_season_and_dates(file_path):
    """Return (season_label, date_range_string) from a pywapor-style filename."""
    name = os.path.basename(file_path).replace(".tif", "")
    parts = name.split("_")
    # Find parts that look like dates (YYYY-MM-DD)
    date_parts = [p for p in parts if len(p) == 10 and p[4] == "-" and p[7] == "-"]
    if len(date_parts) >= 2:
        season = "season1"
        date_range = f"{date_parts[0]} to {date_parts[1]}"
    elif len(date_parts) == 1:
        season = "season1"
        date_range = date_parts[0]
    else:
        season = "season1"
        date_range = name
    return season, date_range

def extract_date_label(file_path):
    _, date_range = extract_season_and_dates(file_path)
    return date_range

def open_clean_raster(file_path, strict_pos=False):
    ds  = rioxarray.open_rasterio(file_path)
    arr = ds.astype("float32")
    nd  = ds.rio.nodata
    if nd is not None:
        arr = arr.where(arr != nd)
    arr = arr.where(np.isfinite(arr))
    if strict_pos:
        arr = arr.where(arr > 0)
    return ds, arr

def save_array_like(template_ds, array2d, out_path, nodata=-9999):
    out = template_ds.copy()
    out.values[0] = np.where(np.isfinite(array2d), array2d, nodata)
    out.rio.write_nodata(nodata, inplace=True)
    out.rio.to_raster(out_path)

# --- Statistics ---------------------------------------------------------------

def percentile_thresholds(arr_da, p_hot=10, p_bright=90):
    v = arr_da.values
    return float(np.nanpercentile(v, p_hot)), float(np.nanpercentile(v, p_bright))

def normalized_score(arr, hot_thr, bright_thr):
    denom = bright_thr - hot_thr
    if np.isclose(denom, 0):
        return np.full_like(arr, np.nan, dtype="float32")
    score = (arr - hot_thr) / denom
    return np.clip(score, 0, 1).astype("float32")

def normalize_weights(w):
    total = sum(w.values())
    if total <= 0:
        raise ValueError("Sum of weights must be > 0.")
    return {k: v / total for k, v in w.items()}

# --- Plotting -----------------------------------------------------------------

def spatial_extent_from_ds(ds):
    return (float(ds.x.min()), float(ds.x.max()),
            float(ds.y.min()), float(ds.y.max()))

def plot_gap_map(arr2d, title, label, extent, cmap="RdYlGn_r"):
    """Single-panel map for a gap raster."""
    fig, ax = plt.subplots(figsize=(10, 6))
    masked = np.ma.masked_invalid(arr2d)
    img = ax.imshow(masked, cmap=cmap, extent=extent, origin="upper")
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.08)
    fig.colorbar(img, cax=cax, label=label)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Longitude (°E)", fontsize=11)
    ax.set_ylabel("Latitude (°N)", fontsize=11)
    plt.tight_layout()
    plt.show()

def plot_target_scatter_hist(arr1d_x, arr1d_y, target_x, target_y,
                              title_scatter, xlabel_s, ylabel_s,
                              title_hist, xlabel_h, percentile=90):
    """Scatter + histogram panel (from Module_4_ProductivityGaps style)."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.scatter(arr1d_x, arr1d_y, marker="*", color="grey", s=10, alpha=0.4)
    ax1.scatter(target_x, target_y, marker="*", color="black", s=120,
                label="Target", zorder=5)
    ax1.axvline(target_x, color="#EE6666", linestyle="--", linewidth=1)
    ax1.axhline(target_y, color="#EE6666", linestyle="--", linewidth=1)
    ax1.set_title(title_scatter, fontsize=12)
    ax1.set_xlabel(xlabel_s, fontsize=11)
    ax1.set_ylabel(ylabel_s, fontsize=11)
    ax1.legend(fontsize=10)

    counts, bins, patches = ax2.hist(arr1d_y, bins=80, facecolor="skyblue",
                                      edgecolor="none")
    lower = float(np.nanpercentile(arr1d_y, 10))
    upper = target_y
    for patch, left, right in zip(patches, bins[:-1], bins[1:]):
        if right < lower:
            patch.set_facecolor("#EE6666")
        elif left >= upper:
            patch.set_facecolor("green")

    handles = [Rectangle((0,0),1,1,color=c,ec="k") for c in ["#EE6666","green"]]
    labels  = ["0–10 Percentile", f">{percentile} Percentile"]
    fakeLine = plt.Line2D([0],[0], color="#EE6666", linestyle="--")
    ax2.legend(handles + [fakeLine], labels + [f"{percentile}th pctl"], fontsize=9)
    ax2.set_title(title_hist, fontsize=12)
    ax2.set_xlabel(xlabel_h, fontsize=11)
    ax2.set_ylabel("Number of pixels", fontsize=11)
    plt.tight_layout()
    plt.show()

def plot_bright_hot_panels(arrays, targets, titles, labels, units,
                            high_is_good, season_label,
                            spot_type="Bright",
                            extent=None, cmap_ind="viridis"):
    """
    Panel map for bright or hot spot identification.

    arrays      : list of 2-D numpy arrays (one per indicator)
    targets     : list of threshold values (one per indicator)
    titles      : list of indicator names
    labels      : list of colorbar labels
    units       : list of unit strings (for the colorbar)
    high_is_good: list of bool – True = bright means >= target, hot means <= target
    spot_type   : "Bright" or "Hot"
    extent      : (xmin, xmax, ymin, ymax)
    """
    n = len(arrays)
    fig, axes = plt.subplots(1, n + 1, figsize=(5*(n+1), 5))

    if spot_type == "Bright":
        spot_color = "green"
        other_color = "lightgrey"
        spot_label = "Bright spot"
        op_symbol = "≥"
    else:
        spot_color = "#CC2222"
        other_color = "lightgrey"
        spot_label = "Hot spot"
        op_symbol = "≤"

    masks = []
    for i, (arr, tgt, title, label, his_good) in enumerate(
            zip(arrays, targets, titles, labels, high_is_good)):
        ax = axes[i]
        masked_arr = np.ma.masked_invalid(arr)

        # Continuous background
        img = ax.imshow(masked_arr, cmap=cmap_ind, extent=extent, origin="upper", alpha=0.6)
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="4%", pad=0.06)
        fig.colorbar(img, cax=cax, label=label)

        # Spot overlay
        if his_good:
            spot_mask = (arr >= tgt) & np.isfinite(arr)
        else:
            spot_mask = (arr <= tgt) & np.isfinite(arr)
        masks.append(spot_mask)

        overlay = np.where(spot_mask, 1.0, np.nan)
        cmap2 = colors.ListedColormap([spot_color])
        ax.imshow(overlay, cmap=cmap2, extent=extent, origin="upper", alpha=0.7,
                  vmin=0.5, vmax=1.5)

        ax.set_title(f"{title}\n{op_symbol} {tgt:.3g} {label} ({spot_label})",
                     fontsize=9)
        ax.set_xlabel("Longitude (°E)", fontsize=9)
        ax.set_ylabel("Latitude (°N)", fontsize=9)

        pct = 100 * np.sum(spot_mask) / np.sum(np.isfinite(arr)) if np.any(np.isfinite(arr)) else 0
        ax.text(0.02, 0.02, f"{pct:.1f}% of valid area",
                transform=ax.transAxes, fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

    # Intersection panel
    ax_int = axes[-1]
    intersection = masks[0].copy()
    for m in masks[1:]:
        intersection &= m
    pct_int = 100 * np.sum(intersection) / np.sum(np.isfinite(arrays[0])) if np.any(np.isfinite(arrays[0])) else 0

    cmap_int = colors.ListedColormap([other_color, spot_color])
    norm_int  = colors.BoundaryNorm([0, 0.5, 1.5], cmap_int.N)
    ax_int.imshow(intersection.astype(float), cmap=cmap_int, norm=norm_int,
                  extent=extent, origin="upper")
    ax_int.set_title(f"Intersection – all indicators\n({spot_label}: {pct_int:.1f}% of area)",
                     fontsize=9)
    ax_int.set_xlabel("Longitude (°E)", fontsize=9)
    ax_int.set_ylabel("Latitude (°N)", fontsize=9)

    fig.suptitle(f"{spot_type} spots — {season_label}", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
    return intersection


## 5. Collect file handles and verify availability

In [ ]:
yield_fhs      = sorted_tifs(yield_folder)
cwp_fhs        = sorted_tifs(cwp_folder)
adequacy_fhs   = sorted_tifs(adequacy_folder)
beneficial_fhs = sorted_tifs(beneficial_folder)

print(f"Yield files              : {len(yield_fhs)}")
print(f"cWP files                : {len(cwp_fhs)}")
print(f"Adequacy files           : {len(adequacy_fhs)}")
print(f"Beneficial fraction files: {len(beneficial_fhs)}")

if not yield_fhs or not cwp_fhs:
    raise FileNotFoundError("Yield and/or cWP rasters were not found. Check dir_data.")

if len(yield_fhs) != len(cwp_fhs):
    raise ValueError("Yield and cWP file counts do not match.")

weighted_ready = (len(yield_fhs) == len(cwp_fhs) ==
                  len(adequacy_fhs) == len(beneficial_fhs))
print(f"\nMulti-indicator analysis ready: {weighted_ready}")
if not weighted_ready:
    print("  (Adequacy / Beneficial fraction data missing or count mismatch — "
          "multi-indicator section will be skipped)")


## 6. Calculate target productivity

Targets are derived as the **{bright_percentile}th percentile** of the seasonal rasters
(or user-supplied manual values when `target_mode = "manual"`).

A scatter plot (Yield vs cWP) and histogram (cWP distribution) are shown for each season.


In [ ]:
target_rows = []

for yield_fh, cwp_fh in zip(yield_fhs, cwp_fhs):
    yield_ds, yield_da = open_clean_raster(yield_fh, strict_positive_mask)
    _,        cwp_da   = open_clean_raster(cwp_fh,   strict_positive_mask)

    Yield_1D = yield_da.values.ravel()
    CWP_1D   = cwp_da.values.ravel()

    pct_target_yield = float(np.nanpercentile(Yield_1D, bright_percentile))
    pct_target_cwp   = float(np.nanpercentile(CWP_1D,   bright_percentile))
    hot_yield        = float(np.nanpercentile(Yield_1D, hot_percentile))
    hot_cwp          = float(np.nanpercentile(CWP_1D,   hot_percentile))

    target_yield = pct_target_yield if target_mode == "percentile" else manual_target_yield
    target_cwp   = pct_target_cwp   if target_mode == "percentile" else manual_target_cwp

    season, date_range = extract_season_and_dates(yield_fh)

    # --- Scatter + histogram plot (Module_4_ProductivityGaps style) ----------
    plot_target_scatter_hist(
        Yield_1D, CWP_1D,
        pct_target_yield, pct_target_cwp,
        title_scatter = f"Crop yield vs cWP – {season} ({date_range})",
        xlabel_s = "Crop yield [ton/ha]",
        ylabel_s = "cWP [kg/m³]",
        title_hist = f"cWP distribution – {season} ({date_range})",
        xlabel_h = "cWP [kg/m³]",
        percentile = bright_percentile,
    )

    target_rows.append({
        "Season"           : season,
        "Period"           : date_range,
        "TargetMode"       : target_mode,
        "TargetYield"      : round(float(target_yield), 3) if target_yield else np.nan,
        "TargetYield_pctl" : round(pct_target_yield, 3),
        "HotYield_pctl"    : round(hot_yield, 3),
        "TargetCWP"        : round(float(target_cwp), 3) if target_cwp else np.nan,
        "TargetCWP_pctl"   : round(pct_target_cwp, 3),
        "HotCWP_pctl"      : round(hot_cwp, 3),
    })

targets_df = pd.DataFrame(target_rows)
display(targets_df)

targets_df.to_csv(os.path.join(out_targets, "TargetYield_TargetCWP.csv"), index=False)
print("Saved targets CSV.")


## 7. Crop yield gaps and crop water productivity gaps

Gap = Target – Actual (only where Actual < Target; elsewhere NaN).  
Maps are saved as GeoTIFF and displayed.


In [ ]:
gap_rows = []

for row, yield_fh, cwp_fh in zip(target_rows, yield_fhs, cwp_fhs):
    yield_ds, yield_da = open_clean_raster(yield_fh, strict_positive_mask)
    _,        cwp_da   = open_clean_raster(cwp_fh,   strict_positive_mask)

    Yield = yield_da.values[0]
    CWP   = cwp_da.values[0]

    target_yield = row["TargetYield"]
    target_cwp   = row["TargetCWP"]
    season       = row["Season"]
    date_range   = row["Period"]
    safe_date    = date_range.replace(" ", "_").replace("/", "-")

    extent = spatial_extent_from_ds(yield_ds)

    # --- Yield gap -----------------------------------------------------------
    Yieldgap = np.where(np.isfinite(Yield) & (Yield < target_yield),
                        target_yield - Yield, np.nan).astype("float32")

    yg_out = os.path.join(out_gaps, f"YieldGap_{safe_date}.tif")
    save_array_like(yield_ds, Yieldgap, yg_out)

    plot_gap_map(Yieldgap,
                 title=f"Crop yield gap [ton/ha] – {season} ({date_range})",
                 label="Yield gap [ton/ha/season]",
                 extent=extent, cmap="RdYlGn_r")

    # --- cWP gap -------------------------------------------------------------
    CWPgap = np.where(np.isfinite(CWP) & (CWP < target_cwp),
                      target_cwp - CWP, np.nan).astype("float32")

    cwpg_out = os.path.join(out_gaps, f"CWPGap_{safe_date}.tif")
    save_array_like(yield_ds, CWPgap, cwpg_out)

    plot_gap_map(CWPgap,
                 title=f"Crop WP gap [kg/m³] – {season} ({date_range})",
                 label="cWP gap [kg/m³]",
                 extent=extent, cmap="jet_r")

    gap_rows.append({
        "Season"        : season,
        "Period"        : date_range,
        "TargetYield"   : round(target_yield, 3),
        "MeanYieldGap"  : round(float(np.nanmean(Yieldgap)), 3),
        "MaxYieldGap"   : round(float(np.nanmax(Yieldgap)), 3) if np.any(np.isfinite(Yieldgap)) else np.nan,
        "TargetCWP"     : round(target_cwp, 3),
        "MeanCWPGap"    : round(float(np.nanmean(CWPgap)), 3),
        "MaxCWPGap"     : round(float(np.nanmax(CWPgap)), 3) if np.any(np.isfinite(CWPgap)) else np.nan,
    })

gaps_df = pd.DataFrame(gap_rows)
display(gaps_df)
gaps_df.to_csv(os.path.join(out_gaps, "YieldCWP_Gaps_summary.csv"), index=False)
print("Saved gaps CSV.")


## 8. Bright and hot spots – crop yield & cWP only (Approach i)

Targets = percentile thresholds computed from each raster.

**Bright spot**: pixel where BOTH yield ≥ target AND cWP ≥ target.  
**Hot spot**: pixel where BOTH yield ≤ hot-threshold AND cWP ≤ hot-threshold.

Three panels are shown per season: yield criterion | cWP criterion | intersection.


In [ ]:
crop_spot_rows = []

for row, yield_fh, cwp_fh in zip(target_rows, yield_fhs, cwp_fhs):
    yield_ds, yield_da = open_clean_raster(yield_fh, strict_positive_mask)
    _,        cwp_da   = open_clean_raster(cwp_fh,   strict_positive_mask)

    Yield = yield_da.values[0]
    CWP   = cwp_da.values[0]

    y_hot, y_bright = percentile_thresholds(yield_da, hot_percentile, bright_percentile)
    c_hot, c_bright = percentile_thresholds(cwp_da,   hot_percentile, bright_percentile)

    season     = row["Season"]
    date_range = row["Period"]
    safe_date  = date_range.replace(" ", "_").replace("/", "-")
    season_lbl = f"{season} ({date_range})"

    extent = spatial_extent_from_ds(yield_ds)

    # --- BRIGHT spots --------------------------------------------------------
    bright_intersection = plot_bright_hot_panels(
        arrays       = [Yield, CWP],
        targets      = [y_bright, c_bright],
        titles       = ["Crop Yield", "Crop WP"],
        labels       = ["ton/ha", "kg/m³"],
        units        = ["ton/ha", "kg/m³"],
        high_is_good = [True, True],
        season_label = season_lbl,
        spot_type    = "Bright",
        extent       = extent,
        cmap_ind     = "YlGn",
    )

    save_array_like(yield_ds, bright_intersection.astype("float32"),
                    os.path.join(out_crop_spots, f"BrightSpots_Crop_{safe_date}.tif"))

    # --- HOT spots -----------------------------------------------------------
    hot_intersection = plot_bright_hot_panels(
        arrays       = [Yield, CWP],
        targets      = [y_hot, c_hot],
        titles       = ["Crop Yield", "Crop WP"],
        labels       = ["ton/ha", "kg/m³"],
        units        = ["ton/ha", "kg/m³"],
        high_is_good = [False, False],
        season_label = season_lbl,
        spot_type    = "Hot",
        extent       = extent,
        cmap_ind     = "YlOrRd",
    )

    save_array_like(yield_ds, hot_intersection.astype("float32"),
                    os.path.join(out_crop_spots, f"HotSpots_Crop_{safe_date}.tif"))

    valid_pixels = int(np.sum(np.isfinite(Yield) & np.isfinite(CWP)))
    crop_spot_rows.append({
        "Season"           : season,
        "Period"           : date_range,
        "YieldBrightThr"   : round(y_bright, 3),
        "CWPBrightThr"     : round(c_bright, 3),
        "BrightPixels"     : int(np.sum(bright_intersection)),
        "BrightPct"        : round(100*np.sum(bright_intersection)/valid_pixels, 2) if valid_pixels else 0,
        "YieldHotThr"      : round(y_hot, 3),
        "CWPHotThr"        : round(c_hot, 3),
        "HotPixels"        : int(np.sum(hot_intersection)),
        "HotPct"           : round(100*np.sum(hot_intersection)/valid_pixels, 2) if valid_pixels else 0,
    })

crop_spots_df = pd.DataFrame(crop_spot_rows)
display(crop_spots_df)
crop_spots_df.to_csv(os.path.join(out_crop_spots, "CropSpots_summary.csv"), index=False)


## 9. Bright and hot spots – weighted multi-indicator (Approach ii)

Inputs: crop yield, cWP, adequacy, beneficial fraction.  
Each indicator is normalised to [0, 1] using its own percentile thresholds,
then combined into a **composite score** using the user-defined weights.

A bright/hot spot is identified based on the composite score percentile.

Panels show each indicator criterion separately, plus the intersection of all four.


In [ ]:
if not weighted_ready:
    print("Weighted analysis skipped – see file availability check in Section 5.")
else:
    w_norm = normalize_weights(weights)
    print("Normalised weights:", {k: round(v, 3) for k, v in w_norm.items()})

    weighted_rows = []

    for row, yield_fh, cwp_fh, adequacy_fh, beneficial_fh in zip(
            target_rows, yield_fhs, cwp_fhs, adequacy_fhs, beneficial_fhs):

        base_ds, yield_da     = open_clean_raster(yield_fh,      strict_positive_mask)
        _,       cwp_da       = open_clean_raster(cwp_fh,        strict_positive_mask)
        _,       adequacy_da  = open_clean_raster(adequacy_fh,   strict_positive_mask)
        _,       beneficial_da= open_clean_raster(beneficial_fh, strict_positive_mask)

        Y = yield_da.values[0]
        C = cwp_da.values[0]
        A = adequacy_da.values[0]
        B = beneficial_da.values[0]

        season     = row["Season"]
        date_range = row["Period"]
        safe_date  = date_range.replace(" ", "_").replace("/", "-")
        season_lbl = f"{season} ({date_range})"

        extent = spatial_extent_from_ds(base_ds)

        # Per-indicator percentile thresholds
        y_hot, y_bright = percentile_thresholds(yield_da,      hot_percentile, bright_percentile)
        c_hot, c_bright = percentile_thresholds(cwp_da,        hot_percentile, bright_percentile)
        a_hot, a_bright = percentile_thresholds(adequacy_da,   hot_percentile, bright_percentile)
        b_hot, b_bright = percentile_thresholds(beneficial_da, hot_percentile, bright_percentile)

        # Normalised scores (0 = hot end, 1 = bright end)
        y_score = normalized_score(Y, y_hot, y_bright)
        c_score = normalized_score(C, c_hot, c_bright)
        a_score = normalized_score(A, a_hot, a_bright)
        b_score = normalized_score(B, b_hot, b_bright)

        valid = np.isfinite(y_score) & np.isfinite(c_score) & np.isfinite(a_score) & np.isfinite(b_score)

        composite = (w_norm["yield"] * y_score +
                     w_norm["cwp"]   * c_score +
                     w_norm["adequacy"] * a_score +
                     w_norm["beneficial_fraction"] * b_score)
        composite = np.where(valid, composite, np.nan)

        comp_bright_thr = float(np.nanpercentile(composite, bright_percentile))
        comp_hot_thr    = float(np.nanpercentile(composite, hot_percentile))

        # Save composite score
        save_array_like(base_ds, composite.astype("float32"),
                        os.path.join(out_weighted_spots, f"CompositeScore_{safe_date}.tif"))

        # --- BRIGHT spots panel (per-indicator + intersection) ---------------
        bright_intersection = plot_bright_hot_panels(
            arrays       = [Y, C, A, B],
            targets      = [y_bright, c_bright, a_bright, b_bright],
            titles       = ["Crop Yield", "Crop WP", "Adequacy", "Beneficial Fraction"],
            labels       = ["ton/ha", "kg/m³", "[-]", "[-]"],
            units        = ["ton/ha", "kg/m³", "", ""],
            high_is_good = [True, True, True, True],
            season_label = season_lbl,
            spot_type    = "Bright",
            extent       = extent,
            cmap_ind     = "YlGn",
        )

        save_array_like(base_ds, bright_intersection.astype("float32"),
                        os.path.join(out_weighted_spots, f"BrightSpots_Weighted_{safe_date}.tif"))

        # Also show composite score bright spots
        comp_bright_mask = valid & (composite >= comp_bright_thr)
        print(f"  Composite-score bright spots ({season}): "
              f"{np.sum(comp_bright_mask)} px  "
              f"({100*np.sum(comp_bright_mask)/np.sum(valid):.1f}% of valid)")

        # --- HOT spots panel (per-indicator + intersection) ------------------
        hot_intersection = plot_bright_hot_panels(
            arrays       = [Y, C, A, B],
            targets      = [y_hot, c_hot, a_hot, b_hot],
            titles       = ["Crop Yield", "Crop WP", "Adequacy", "Beneficial Fraction"],
            labels       = ["ton/ha", "kg/m³", "[-]", "[-]"],
            units        = ["ton/ha", "kg/m³", "", ""],
            high_is_good = [False, False, False, False],
            season_label = season_lbl,
            spot_type    = "Hot",
            extent       = extent,
            cmap_ind     = "YlOrRd",
        )

        save_array_like(base_ds, hot_intersection.astype("float32"),
                        os.path.join(out_weighted_spots, f"HotSpots_Weighted_{safe_date}.tif"))

        comp_hot_mask = valid & (composite <= comp_hot_thr)
        print(f"  Composite-score hot spots ({season}): "
              f"{np.sum(comp_hot_mask)} px  "
              f"({100*np.sum(comp_hot_mask)/np.sum(valid):.1f}% of valid)")

        valid_px = int(np.sum(valid))
        weighted_rows.append({
            "Season"              : season,
            "Period"              : date_range,
            "WeightYield"         : round(w_norm["yield"], 3),
            "WeightCWP"           : round(w_norm["cwp"], 3),
            "WeightAdequacy"      : round(w_norm["adequacy"], 3),
            "WeightBenFrac"       : round(w_norm["beneficial_fraction"], 3),
            "CompBrightThr"       : round(comp_bright_thr, 3),
            "CompHotThr"          : round(comp_hot_thr, 3),
            "BrightPixels_Ind"    : int(np.sum(bright_intersection)),
            "BrightPct_Ind"       : round(100*np.sum(bright_intersection)/valid_px,2) if valid_px else 0,
            "HotPixels_Ind"       : int(np.sum(hot_intersection)),
            "HotPct_Ind"          : round(100*np.sum(hot_intersection)/valid_px,2) if valid_px else 0,
        })

    weighted_df = pd.DataFrame(weighted_rows)
    display(weighted_df)
    weighted_df.to_csv(os.path.join(out_weighted_spots, "WeightedSpots_summary.csv"), index=False)
    print("Saved weighted spots CSV.")


## 10. Download results

In [ ]:
zip_out = "/content/Module_4_BrightHotSpots_Yield_CWP_IPA_Gaps_outputs.zip"
!zip -r $zip_out /content/output/ > /dev/null
print("Archive created:", zip_out)

from google.colab import files
files.download(zip_out)
